## Gymnasium — The Standard RL Environment Interface

This notebook introduces [Gymnasium](https://gymnasium.farama.org/), the standard Python library for reinforcement learning environments.
It covers the core API, the most important built-in environments, and shows how to implement a **custom environment** that you can use to test your own RL agents.

Gymnasium was originally developed by OpenAI as *Gym* and is now maintained by the [Farama Foundation](https://farama.org/).

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

## 1. Why Gymnasium?

A central challenge in RL research is **comparability**: results from different papers are only meaningful if the agents were trained on identical environments.
Gymnasium solves this by providing:

- A **standardised API** (`reset`, `step`, `render`, `close`) that every environment implements
- A large collection of **built-in benchmark environments**
- A framework for building and registering **custom environments**

Because the API is fixed, an RL algorithm written for one environment works on any other without code changes — only the network input/output dimensions differ.

**Built-in environment categories:**

| Category | Examples | Notes |
|---|---|---|
| Classic Control | CartPole, MountainCar, Pendulum | Low-dimensional, fast — ideal for learning |
| Toy Text | FrozenLake, Blackjack, Taxi | Discrete state/action — good for tabular methods |
| Box2D | LunarLander, BipedalWalker | 2D physics, requires `pip install gymnasium[box2d]` |
| MuJoCo | Ant, HalfCheetah, Humanoid | High-dimensional continuous control, physics engine |
| Atari | Pong, Breakout, ... | Pixel observations, requires `pip install gymnasium[atari]` |

**Notable third-party environments:**
- [Highway-env](https://github.com/Farama-Foundation/HighwayEnv) — autonomous driving decisions
- [Panda-gym](https://github.com/qgallouedec/panda-gym) — industrial robot arm control
- [MiniGrid](https://github.com/Farama-Foundation/MiniGrid) — customisable 2D grid worlds

## 2. The Core API

Every Gymnasium environment exposes the same five methods:

| Method | Description |
|---|---|
| `env = gym.make(id)` | Load an environment by name |
| `obs, info = env.reset()` | Start a new episode, returns the initial observation |
| `obs, reward, terminated, truncated, info = env.step(action)` | Apply one action, advance the simulation by one timestep |
| `env.render()` | Visualise the current state |
| `env.close()` | Release resources |

The `step()` return values:
- **obs** — the new observation after the action
- **reward** — scalar feedback signal
- **terminated** — `True` if the agent reached a terminal state (goal or failure)
- **truncated** — `True` if the episode was cut off by a time limit
- **info** — dict with optional diagnostic data (not used for learning)

An episode ends when `terminated or truncated` is `True`.

## 3. Observation and Action Spaces

Every environment defines two **spaces** that describe the shape and type of observations and actions:

- `env.observation_space` — the space of possible observations
- `env.action_space` — the space of valid actions

The two most common space types are:

**`Discrete(n)`** — a finite set of integers `{0, 1, ..., n-1}`
```
FrozenLake: Discrete(16)   # 16 grid positions
CartPole:   Discrete(2)    # push left or right
```

**`Box(low, high, shape, dtype)`** — a continuous n-dimensional box (like a NumPy array with bounds)
```
CartPole observation:  Box(shape=(4,))   # [cart_pos, cart_vel, pole_angle, pole_vel]
Pendulum action:       Box(shape=(1,))   # continuous torque in [-2, 2]
Atari observation:     Box(shape=(210, 160, 3), dtype=uint8)  # RGB image
```

You can always sample a random valid action with `env.action_space.sample()`.

In [ ]:
env = gym.make('CartPole-v1')

print('Observation space:', env.observation_space)
print('  shape:', env.observation_space.shape)
print('  low:  ', env.observation_space.low)
print('  high: ', env.observation_space.high)
print()
print('Action space:     ', env.action_space)
print('  n actions:      ', env.action_space.n)
print()
print('Sample observation:', env.observation_space.sample())
print('Sample action:     ', env.action_space.sample())

env.close()

## 4. First Steps: FrozenLake

FrozenLake is a simple grid-world environment:
- The agent starts at position S and must reach goal G
- H tiles are holes — falling in ends the episode with reward 0
- Actions: LEFT (0), DOWN (1), RIGHT (2), UP (3)
- Reward: +1 only on reaching G

```
S F F F
F H F H
F F F H
H F F G
```

In [ ]:
import gymnasium as gym
import time

# is_slippery=False makes the environment deterministic — the agent goes where it intends
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
obs, info = env.reset()

print('action space:      ', env.action_space)
print('observation space: ', env.observation_space)
print('initial observation:', obs)

In [ ]:
LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3

# Execute one step manually
obs, reward, terminated, truncated, info = env.step(DOWN)
env.render()

print('new observation:', obs)
print('reward:         ', reward)
print('terminated:     ', terminated)
print('truncated:      ', truncated)
print('info:           ', info)

In [ ]:
env.close()

## 5. The Standard Episode Loop

The canonical way to interact with a Gymnasium environment is the **episode loop**.
It resets the environment at the start, then calls `step()` until the episode ends.

In [ ]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)

n_episodes = 3
for episode in range(n_episodes):
    obs, info = env.reset()
    done = False
    total_reward = 0
    steps = 0

    while not done:
        action = env.action_space.sample()  # random agent
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    print(f'Episode {episode + 1}: {steps} steps, total reward = {total_reward}')

env.close()

## 6. Building a Custom Environment

Any Gymnasium environment is a Python class that inherits from `gym.Env` and implements four methods:

| Method | What to implement |
|---|---|
| `__init__` | Define `self.observation_space` and `self.action_space` |
| `reset` | Initialise state, return `(obs, info)` |
| `step` | Apply action, update state, return `(obs, reward, terminated, truncated, info)` |
| `render` | Optional visualisation |

The example below implements a minimal **1D corridor** environment:
- The agent starts at position 0 in a corridor of length `n`
- Actions: move LEFT (0) or RIGHT (1)
- Goal: reach the rightmost position (`n - 1`)
- Reward: +1 on reaching the goal, -0.01 at every other step (step cost)
- Episode ends when the goal is reached or after `max_steps` steps

In [ ]:
class CorridorEnv(gym.Env):
    """A simple 1D corridor: agent starts at 0 and must reach position n-1."""

    metadata = {'render_modes': ['human']}

    def __init__(self, n=6, max_steps=50):
        super().__init__()
        self.n = n
        self.max_steps = max_steps

        # One integer observation: the current position
        self.observation_space = gym.spaces.Discrete(n)

        # Two actions: 0 = left, 1 = right
        self.action_space = gym.spaces.Discrete(2)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)       # handles the RNG seed
        self.pos = 0
        self.steps_taken = 0
        return self.pos, {}

    def step(self, action):
        if action == 0:                # move left
            self.pos = max(0, self.pos - 1)
        else:                          # move right
            self.pos = min(self.n - 1, self.pos + 1)

        self.steps_taken += 1
        terminated = self.pos == self.n - 1
        truncated  = self.steps_taken >= self.max_steps
        reward     = 1.0 if terminated else -0.01

        return self.pos, reward, terminated, truncated, {}

    def render(self):
        corridor = ['.'] * self.n
        corridor[0]        = 'S'      # start
        corridor[-1]       = 'G'      # goal
        corridor[self.pos] = 'A'      # agent (overwrites S/G if on them)
        print(' '.join(corridor))


# Quick smoke test
env = CorridorEnv(n=6)
obs, _ = env.reset()
env.render()

for _ in range(7):
    obs, reward, terminated, truncated, _ = env.step(1)  # always go right
    env.render()
    if terminated or truncated:
        print(f'Done! Final reward: {reward}')
        break

### Running the custom environment with a random agent

The custom environment is fully compatible with the standard episode loop — no changes needed:

In [ ]:
env = CorridorEnv(n=6, max_steps=30)

rewards_per_episode = []
for episode in range(20):
    obs, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated

    rewards_per_episode.append(total_reward)

plt.plot(rewards_per_episode, marker='o')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('Random agent on CorridorEnv')
plt.grid(True)
plt.show()

### Exercise: Extend the CorridorEnv

Modify `CorridorEnv` so that:
1. The agent starts at a **random position** instead of always at 0 (use `self.np_random.integers(...)` — the seeded RNG provided by `gym.Env`)
2. There is a **hole** at position `n // 2` — stepping on it ends the episode with reward -1

Test your environment with the random agent above.

In [ ]:
# YOUR CODE HERE


## 7. Rendering as Video (Colab)

Google Colab does not support the standard `render_mode='human'` (pygame window). The workaround is to use `render_mode='rgb_array'`, collect frames, and display them as an inline video.

**Note:** this approach also works locally but the video quality is lower than a pygame window.

In [ ]:
import imageio

# is_slippery=False makes the environment deterministic
env = gym.make('FrozenLake-v1', render_mode='rgb_array', is_slippery=False)

frames = []
obs, info = env.reset()
done = False

while not done:
    frame = env.render()          # returns an (H, W, 3) numpy array
    frames.append(frame)

    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

# Capture the final frame
frames.append(env.render())
env.close()

print(f'Captured {len(frames)} frames, shape: {frames[0].shape}')

In [ ]:
video_path = './frozen_lake_vid.mp4'
imageio.mimsave(video_path, frames, fps=5)

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_path, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")